In [ ]:
import sys, os

# Locate config.py by walking up from the current working directory.
# Works regardless of where Jupyter was launched from.
_search = os.getcwd()
for _ in range(6):
    if os.path.isfile(os.path.join(_search, 'config.py')):
        sys.path.insert(0, _search)
        break
    _search = os.path.dirname(_search)

import config
# ======================================================================


import pandas as pd
import pyodbc

# Collect PO Pending Raw
po_file_path = {
    'Access_PO': config.CLEANED_PO_PENDING_PATH,
}

# Load excel to DF from the specified sheet name
def load_data(po_file_path):
    dataframes = {}
    for key, path in po_file_path.items():
        print(f"Loading file for {key}: {path}")
        dataframes[key] = pd.read_excel(path, sheet_name='cleaned data')
    return dataframes


# clean data for Access_PO
def clean_access_po(df):
    # Exclude unnecessary columns
    df['PO Cartons'] = df['PO_Qty'] / df['PC_Cartons']
    df = df.drop(columns=['Devision','Unit','Customer', 'Actual_Qty'])
    # Rename Column
    df.rename(columns={
        'PO Num': 'SHM PO No.',
        'PO Ref': 'PO CJ No.',
        'DC_Name': 'Ship to DC',
        'Rec_Date': 'Delivery Date',
        'CJ_Description':'Product Name'
    },inplace=True)
    return df


# Connect to Access database and fetch product details  
def load_access_data():  
    access_db_path = config.ACCESS_DB_PATH  
    conn_str = (  
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'  
        f'DBQ={access_db_path};'  
    )  
    try:  
        conn = pyodbc.connect(conn_str)  
        print("Connection successful")

        query = """  
        SELECT CJ_Item,  
               SHM_Item,
               [Supplier Name],
               Devision
          FROM qry_Product_List  
        """

        access_df = pd.read_sql(query, conn)
        conn.close()  
        print("Connection to Access database is closed.")  
        return access_df
    except Exception as e:
        print(f"Error: {e}")
        return None
    
# Load DataFrame
dataframes = load_data(po_file_path)
cleaned_dataframes = {
    'Access_PO': clean_access_po(dataframes['Access_PO'])
}
# Append DataFrames together
final_df = pd.concat(cleaned_dataframes.values(), ignore_index=True)
final_df['CJ_Item'] = final_df['CJ_Item'].astype(str)
final_df['SHM_Item'] = final_df['SHM_Item'].astype(str)

# Load Access data
access_df = load_access_data()
if access_df is not None:
    # Convert CJ_Item in access_df to string
    access_df['CJ_Item'] = access_df['CJ_Item'].astype(str)
    access_df['SHM_Item'] = access_df['SHM_Item'].astype(str)

    # Merge DataFrames
    merged_df = pd.merge(final_df, access_df, on=['CJ_Item', 'SHM_Item'], how='left', suffixes=('', '_access'))

    # Update Supplier Name in final_df where it is null
    final_df['Supplier Name'] = final_df['Supplier Name'].fillna(merged_df['Supplier Name_access'])
    final_df['Division'] = merged_df['Devision']

# Load OwnerSCM
owner_scm_file_path = config.MASTER_LEADTIME_PATH
owner_scm_df = pd.read_excel(owner_scm_file_path, sheet_name='All_Product', header=1)

# Cast column CJ_Item to str
owner_scm_df['CJ_Item'] = owner_scm_df['CJ_Item'].astype(str)
# Select some columns
owner_scm_df_selected = owner_scm_df[['SHM_Item','CJ_Item','OwnerSCM','Base Lead Time (Days)']]
owner_scm_df_selected['CJ_Item'] = owner_scm_df_selected['CJ_Item'].fillna('')
owner_scm_df_selected['CJ_Item'] = owner_scm_df_selected['CJ_Item'].apply(lambda x: x.split('.')[0] if '.0' in str(x) else x)

# Merge with Master Leadtime for get OwnerSCM
final_df = pd.merge(final_df,owner_scm_df_selected[['CJ_Item','SHM_Item','OwnerSCM']],on=['CJ_Item','SHM_Item'],how='left')

# Arrange column
desire_order = [
    'Division',
    'OwnerSCM',
    'PO Date',
    'SHM PO No.',
    'Supplier Name',
    'SHM_Item',
    'CJ_Item',
    'Product Name',
    'PO CJ No.',
    'PC_Cartons',
    'Ship to DC',
    'PO Cartons',
    'PO_Qty',
    'Delivery Date',
    'Delivery_Status'
]
final_df = final_df.reindex(columns=desire_order)

final_df_pivot = final_df.pivot_table(
    index= ['CJ_Item','Ship to DC'],
    values=['Delivery Date'],
    aggfunc ='min'
).reset_index().rename(columns={'Delivery Date':'First Delivery Date'})


# Add new column
final_df_pivot['ConcatIndex'] = final_df_pivot['CJ_Item'] + final_df_pivot['Ship to DC']

# Re-order column
final_df_pivot = final_df_pivot[['CJ_Item', 'Ship to DC', 'First Delivery Date', 'ConcatIndex']]

print(final_df.dtypes)


# Save updated final_df to Excel
save_path = config.RAW_PO_PENDING_PATH


with pd.ExcelWriter(save_path, mode='w') as writer:
    # Save final_df to the "All PO Pending" sheet
    final_df.to_excel(writer, sheet_name=config.RAW_PO_ALL_SHEET, index=False)
    final_df_pivot.to_excel(writer, sheet_name=config.RAW_PO_ETA_SHEET, index=False)

print(f"Combined all po pending report has been saved to: {save_path}")

